# Raptor v5: CoAtNet only

The existing CoAtNet RMLP-2 trunk and original 12-label attention-MIL head.
There is no ResNet branch, fusion block, or fusion gate in the model.
Input remains an existing preprocessed `[B,K,3,384,384]` adjacent-slice bag.
Continue using the same study splits, labels, sampling and normalization as
your current data pipeline. This notebook does not replace preprocessing.

Attach an original Raptor v5 checkpoint, a trained v3 fusion checkpoint, or
an exported CoAtNet-only checkpoint. A fusion checkpoint contributes only its
CoAtNet/head tensors. No model weights are downloaded. Use timm 1.0.22.


In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

try:
    timm_version = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError:
    timm_version = None
if timm_version != '1.0.22':
    wheels = sorted(Path('/kaggle/input').rglob('timm-1.0.22-py3-none-any.whl'))
    if len(wheels) != 1:
        raise RuntimeError('Attach exactly one timm 1.0.22 wheel for offline use')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           '--no-index', '--no-deps', str(wheels[0])])


In [ ]:
"""CoAtNet-only Raptor v5 with the original per-finding attention-MIL head.

Consumes existing preprocessed [B, K, 3, 384, 384] triplet bags. Loading an old
fusion checkpoint keeps its trained CoAtNet/head tensors, without constructing
the ResNet encoder or either fusion block. All checkpoint loads are offline.
"""

from __future__ import annotations

from collections.abc import Mapping
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import torch
from torch import Tensor, nn


COATNET_ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
MODEL_FORMAT = "raptor_v5_coatnet_v1"
LEGACY_FUSED_FORMAT = "raptor_v5_resnet50_v3"
LABELS = (
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
)


@dataclass(frozen=True)
class RaptorConfig:
    head_dropout: float = 0.2
    gradient_checkpointing: bool = False

    def __post_init__(self):
        if not 0 <= self.head_dropout < 1:
            raise ValueError("head_dropout must be in [0, 1)")


class RaptorClassifier(nn.Module):
    """Original Raptor attention-MIL head, retaining its parameter names."""

    def __init__(self, backbone: nn.Module, F_dim: int, n: int = 12, drop: float = 0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x: Tensor) -> Tensor:
        if (x.ndim != 5 or x.shape[0] == 0 or x.shape[1] == 0
                or tuple(x.shape[2:]) != (3, 384, 384)):
            raise ValueError("Raptor expects nonempty [B,K,3,384,384] preprocessed triplet bags")
        b, k = x.shape[:2]
        return self.backbone(x.flatten(0, 1)).reshape(b, k, -1)

    def head(self, feats: Tensor) -> Tensor:
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x: Tensor) -> Tensor:
        return self.head(self.encode(x))


def _checkpoint(source: str | Path | Mapping[str, Any]) -> Mapping[str, Any]:
    ck = source if isinstance(source, Mapping) else torch.load(
        source, map_location="cpu", weights_only=True,
    )
    if not isinstance(ck, Mapping):
        raise ValueError("Expected a checkpoint mapping")
    return ck


def _model_state(ck: Mapping[str, Any], weights: str) -> Mapping[str, Tensor]:
    if weights == "raw":
        state = ck.get("model")
    elif weights == "ema":
        ema = ck.get("ema")
        state = ema.get("shadow") if isinstance(ema, Mapping) else None
        if state is None:
            raise ValueError("Checkpoint has no embedded EMA shadow; use raw for a standalone EMA snapshot")
    else:
        raise ValueError("weights must be 'raw' or 'ema'")
    if not isinstance(state, Mapping) or not state:
        raise ValueError("Checkpoint has no model state dictionary")
    return state


def _coatnet_state(state: Mapping[str, Tensor], legacy_fused: bool) -> dict[str, Tensor]:
    if not legacy_fused:
        return dict(state)
    kept = {}
    for name, value in state.items():
        if name.startswith("backbone.coatnet."):
            kept["backbone." + name.removeprefix("backbone.coatnet.")] = value
        elif name.startswith(("backbone.resnet50.", "backbone.f1.", "backbone.f2.")):
            continue
        elif name.startswith("backbone."):
            raise ValueError(f"Unexpected legacy backbone tensor: {name}")
        else:
            kept[name] = value
    return kept


def load_raptor_checkpoint(
    source: str | Path | Mapping[str, Any], *, weights: str = "raw",
    config: RaptorConfig | None = None,
) -> RaptorClassifier:
    """Load an original parent, a CoAtNet-only checkpoint, or strip v3 fusion.

    ``weights='ema'`` selects a trainer checkpoint's embedded ``ema.shadow``.
    A standalone EMA checkpoint stores its selected weights in ``model`` and
    uses the default ``weights='raw'``. Optimizer state is never migrated.
    """
    import timm

    ck = _checkpoint(source)
    fmt = ck.get("format")
    if fmt not in (None, MODEL_FORMAT, LEGACY_FUSED_FORMAT):
        raise ValueError(f"Unsupported Raptor checkpoint format: {fmt}")
    if ck.get("arch", COATNET_ARCH) != COATNET_ARCH or ck.get("res", 384) != 384:
        raise ValueError("Expected Raptor v5 CoAtNet RMLP-2 at res=384")
    if fmt is not None and any(key not in ck for key in ("arch", "res", "labels")):
        raise ValueError("Formatted checkpoint must declare architecture, resolution and label order")
    if "labels" in ck and list(ck["labels"]) != list(LABELS):
        raise ValueError("Checkpoint label order differs from Raptor v5")
    state = _coatnet_state(_model_state(ck, weights), fmt == LEGACY_FUSED_FORMAT)
    if config is None:
        if fmt == MODEL_FORMAT:
            config = RaptorConfig(**ck["raptor_config"])
        else:
            previous = ck.get("fusion_config", {})
            config = RaptorConfig(**{key: previous[key] for key in
                ("head_dropout", "gradient_checkpointing") if key in previous})
    backbone = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0,
                                 in_chans=3, global_pool="avg")
    backbone.set_grad_checkpointing(config.gradient_checkpointing)
    model = RaptorClassifier(backbone, F_dim=backbone.num_features, drop=config.head_dropout)
    model.load_state_dict(state, strict=True)
    model.raptor_config = config
    model.requires_grad_(True)
    return model


def optimizer_groups(model: RaptorClassifier, backbone_lr: float = 1e-5,
                     head_lr: float = 1e-4) -> list[dict[str, Any]]:
    """CoAtNet and the unchanged MIL head, each trainable parameter exactly once."""
    if not isinstance(model, RaptorClassifier) or not hasattr(model, "raptor_config"):
        raise TypeError("Expected a loaded CoAtNet-only Raptor model")
    if any(not p.requires_grad for p in model.parameters()):
        raise ValueError("CoAtNet and the head must remain unfrozen")
    groups = {"coatnet": [], "head": []}
    for name, parameter in model.named_parameters():
        groups["coatnet" if name.startswith("backbone.") else "head"].append(parameter)
    return [{"name": name, "params": parameters,
             "lr": backbone_lr if name == "coatnet" else head_lr}
            for name, parameters in groups.items()]


def raptor_checkpoint(model: RaptorClassifier) -> dict[str, Any]:
    """Serialize only CoAtNet and the MIL head, with their input/label contract."""
    if not isinstance(model, RaptorClassifier) or not hasattr(model, "raptor_config"):
        raise TypeError("Expected a loaded CoAtNet-only Raptor model")
    return {"format": MODEL_FORMAT, "arch": COATNET_ARCH, "res": 384,
            "labels": list(LABELS), "raptor_config": asdict(model.raptor_config),
            "model": model.state_dict()}


In [ ]:
# Choose the checkpoint whose CoAtNet/head weights you want to retain.
checkpoint_path = Path('/kaggle/input/raptor-knee-maxspan/raptor_ft_coatnet_v5_full_swa.pt')
# 'ema' selects embedded ema.shadow; standalone epoch_*_ema.pt files use 'raw'.
source_weights = 'raw'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_raptor_checkpoint(checkpoint_path, weights=source_weights).to(device)
optimizer = torch.optim.AdamW(optimizer_groups(model), weight_decay=0.01)
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('Optimizer groups:', [(g['name'], g['lr']) for g in optimizer.param_groups])


In [ ]:
# Reuse your existing loader's preprocessed images and targets.
# Keep batch_size=1 for variable-length studies; evaluate all saved windows.
from torch.utils.checkpoint import checkpoint

def bag_logits(triplets, chunk=12):
    # Chunk only the encoder; the unchanged head sees the entire study bag.
    if (triplets.ndim != 5 or triplets.shape[0] != 1 or triplets.shape[1] == 0
            or tuple(triplets.shape[2:]) != (3, 384, 384)):
        raise ValueError('Expected [1,K,3,384,384]')
    if chunk < 1:
        raise ValueError('chunk must be positive')
    features = []
    for windows in triplets[0].split(chunk):
        if model.training and torch.is_grad_enabled():
            features.append(checkpoint(model.backbone, windows, use_reentrant=False))
        else:
            features.append(model.backbone(windows))
    return model.head(torch.cat(features)[None])

use_amp = device.type == 'cuda'
amp_dtype = (torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported()
             else torch.float16)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp and amp_dtype == torch.float16)

def train_step(triplets, targets):
    model.train()
    # Preserve the completed run's frozen BatchNorm running statistics.
    for module in model.modules():
        if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
            module.eval()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
        logits = bag_logits(triplets.to(device))
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets.to(device=device, dtype=torch.float32))
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach())

# Start a new optimizer: the old fusion optimizer has different groups.
# saved = raptor_checkpoint(model)
# saved.update(optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), epoch=epoch)
# torch.save(saved, '/kaggle/working/raptor_v5_coatnet.pt')
